## Module Import

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "numpy", "matplotlib", "seaborn"])

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'matplotlib'

## Manual Series and Dataframe Examples

In [ ]:
example_budgets = pd.Series([10000,100000,100000000], name="budget")

budget_categories = pd.DataFrame({
    "category": ["Low", "Medium", "High", "Very High"],
    "budget_threshold": ["15000000", "100000000", "200000000", "inf"]
})

In [ ]:
example_budgets

In [ ]:
budget_categories

## Data Loading and Inspection

In [ ]:
df_raw = pd.read_csv("../data/raw/movies.csv")

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

In [ ]:
df_raw.isna().sum()

## Data Cleaning

In [ ]:
# Drops rows that have a null value in budget, gross, or runtime (these attributes are impertive to the
# analysis and populating them with 'fake' data could skew the results)
df_drop_budget_gross_runtime = df_raw.dropna(subset=["budget", "gross", "runtime"])

# Fills remaining null values with "Unknown"
df_drop_budget_gross_runtime["company"] = df_drop_budget_gross_runtime["company"].fillna("Unknown")
df_drop_budget_gross_runtime["country"] = df_drop_budget_gross_runtime["country"].fillna("Unknown")
df_drop_budget_gross_runtime["rating"] = df_drop_budget_gross_runtime["rating"].fillna("Unknown")

# Output is 0 confirming there are no longer any null values in the data
df_drop_budget_gross_runtime.isna().sum()

In [ ]:
# Separates the "released" attribute into 2 new attributes "release_date" and "release_country" and removes the original "released" attribute
# released_date -> datetime object  release_country -> string
df_drop_budget_gross_runtime[["release_date", "release_country"]] = df_drop_budget_gross_runtime["released"].str.extract(r"^(.*)\s\((.*)\)$")
df_dropped = df_drop_budget_gross_runtime.drop(columns='released')
df_dropped["release_date"] = pd.to_datetime(df_dropped["release_date"], errors="coerce")

In [ ]:
# Some released attributes are formated in a way that pd.to_datetime cannot parse, so they return NaT
# Below fills in those missing dates with the first day of the proper released year
null_mask = df_dropped["release_date"].isna()

df_dropped.loc[null_mask, "release_date"] = pd.to_datetime(
    pd.DataFrame({
        "year": df_dropped.loc[null_mask, "year"],
        "month": 1,
        "day": 1
    }),
    errors="coerce"
)

In [ ]:
# Ensures all values that are supposed to be numeric get converted to a numeric value
df_dropped["score"] = pd.to_numeric(df_dropped["score"], errors="coerce")
df_dropped["votes"] = pd.to_numeric(df_dropped["votes"], errors="coerce")
df_dropped["budget"] = pd.to_numeric(df_dropped["budget"], errors="coerce")
df_dropped["gross"] = pd.to_numeric(df_dropped["gross"], errors="coerce")
df_dropped["runtime"] = pd.to_numeric(df_dropped["runtime"], errors="coerce")

In [ ]:
# Confirms all values that are supposed to be numeric are
df_dropped.dtypes

In [ ]:
# Created a budget_category attribute that splits movies' budgets into 4 categories
# b < $15 mil -> Low Budget
# $15 mil < b < $100 mil -> Medium Budget
# $100 mil < b < $200 mil -> High Budget
# $200 < b -> Very High Budget
def budget_category(budget):
    if pd.isna(budget):
        return "Unknown"
    elif budget >= 200000000:
        return "Very High"
    elif budget >= 100000000:
        return "High"
    elif budget >= 15000000:
        return "Medium"
    else:
        return "Low"

df_dropped["budget_category"] = df_dropped["budget"].apply(budget_category)

In [ ]:
# Create a roi attribute
df_dropped["roi"] = df_dropped["gross"] / df_dropped["budget"]

In [ ]:
# Final, cleaned data
df = df_dropped.copy()

# First 10 cleaned elements
df.iloc[0:10]

## Cleaned Data Inspection

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
# Exporting Cleaned Dataset
df.to_csv('cleaned_movies_data.csv', index=False)

# Exploratory Data Analysis (EDA)

### Descriptive Statistics
Below we compute key summary statistics for the main numeric columns in the dataset.

In [ ]:
key_cols = ["score", "budget", "gross", "runtime"]

for col in key_cols:
    print(f"--- {col.upper()} ---")
    print(f"  Mean:   {df[col].mean():.2f}")
    print(f"  Median: {df[col].median():.2f}")
    print(f"  Min:    {df[col].min():.2f}")
    print(f"  Max:    {df[col].max():.2f}")
    print(f"  Std:    {df[col].std():.2f}")
    print()

### Descriptive Statistics by Genre
To understand how key metrics differ across movie genres, we group the data by genre and compute summary statistics.
This reveals which genres tend to have higher budgets, better scores, or longer runtimes.

In [ ]:
genre_stats = df.groupby("genre")[["score", "budget", "gross", "runtime"]].agg(
    ["mean", "median", "std", "min", "max", "count"]
)

genre_stats

### Correlation Analysis
We compute a correlation matrix for the key numeric variables to identify which factors move together.

In [ ]:
numeric_cols = ["score", "votes", "budget", "gross", "runtime", "roi"]
corr_matrix = df[numeric_cols].corr()

corr_matrix

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation Matrix of Numeric Features")
fig.tight_layout()
fig.savefig("figures/correlation_heatmap.png", dpi=300)
plt.show()

**Key Takeaways from the Correlation Matrix:**
- **Budget and Gross** have a strong positive correlation, confirming that higher-budget films tend to earn more at the box office.
- **Votes and Gross** are also strongly correlated — movies that earn more tend to attract more audience engagement on IMDb.
- **Score and ROI** show a weaker relationship, suggesting that critical acclaim alone does not guarantee profitability.
- **Runtime** has a modest positive correlation with score, indicating that longer movies tend to be slightly better reviewed.

### Feature Exploration

#### Do certain movie ratings (PG, PG-13, R) have different average budgets and scores?
We compare the three most common MPAA ratings to see if there are meaningful differences in budget and critical reception.

In [ ]:
main_ratings = ["PG", "PG-13", "R"]
rating_df = df[df["rating"].isin(main_ratings)]

rating_summary = rating_df.groupby("rating").agg(
    avg_budget=("budget", "mean"),
    avg_score=("score", "mean"),
    avg_gross=("gross", "mean"),
    movie_count=("name", "count")
).loc[main_ratings]

rating_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(rating_summary.index, rating_summary["avg_budget"], color=["#4C72B0", "#55A868", "#C44E52"])
axes[0].set_title("Average Budget by Rating")
axes[0].set_ylabel("Average Budget ($)")
axes[0].set_xlabel("Rating")
axes[0].ticklabel_format(style="plain", axis="y")

axes[1].bar(rating_summary.index, rating_summary["avg_score"], color=["#4C72B0", "#55A868", "#C44E52"])
axes[1].set_title("Average IMDb Score by Rating")
axes[1].set_ylabel("Average IMDb Score")
axes[1].set_xlabel("Rating")

fig.suptitle("Budget and Score Comparison Across Movie Ratings", fontsize=14)
fig.tight_layout()
fig.savefig("figures/rating_budget_score.png", dpi=300)
plt.show()

**Findings:** PG-13 movies tend to have the highest average budgets, likely because studios invest more in broadly accessible blockbusters. 
R-rated movies, while numerous, have lower average budgets. 
Interestingly, R-rated films have slightly higher average IMDb scores, suggesting that the creative freedom of an R rating may lead to better-reviewed films.

#### Has movie profitability (ROI) changed over time?
We look at how the average return on investment has trended across the decades in our dataset.

In [ ]:
df["decade"] = (df["year"] // 10) * 10

decade_roi = df.groupby("decade").agg(
    avg_roi=("roi", "mean"),
    median_roi=("roi", "median"),
    movie_count=("name", "count")
)

decade_roi

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(decade_roi.index.astype(str), decade_roi["avg_roi"], marker="o", label="Mean ROI", color="#4C72B0")
ax.plot(decade_roi.index.astype(str), decade_roi["median_roi"], marker="s", label="Median ROI", color="#C44E52")
ax.set_title("Movie ROI by Decade")
ax.set_xlabel("Decade")
ax.set_ylabel("Return on Investment")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("figures/roi_by_decade.png", dpi=300)
plt.show()

**Findings:** The median ROI has remained relatively stable across decades, but the mean ROI shows more variation — 
driven by a small number of breakout hits with extremely high returns. 
This gap between mean and median highlights that movie profitability is heavily skewed: most films earn modest returns, while a few outliers generate enormous profits.

# Research Analysis

### Question 1: How had the average rating of R-rated movies changed from 1980-2020?

In [ ]:
# Dataframe with ONLY the R-rated movies
r_movies = df.loc[df["rating"] == "R"]

# Average score and movie count per year for R-rated movies
r_movies_trend = r_movies.groupby("year").agg(
    avg_score=("score", "mean"),
    movie_count=("name", "count")
).sort_index(ascending=True)

In [ ]:
r_movies_trend

### Question 2: Does a certain genre generate high gross revenue?

In [ ]:
# Average gross, total gross, and movie count per genre category
genre_gross_summary = df.groupby("genre").agg(
    avg_gross=("gross", "mean"),
    total_gross=("gross", "sum"),
    movie_count=("name", "count"),
).sort_values("avg_gross", ascending=False)

In [ ]:
genre_gross_summary

### Question 3: How are IMDb scores distributed across all action movies?

In [ ]:
# Dataframe with only action movies
action_movies = df[df["genre"] == "Action"].reset_index()

# Average, median, min and max scores, and movie count for action movies
action_score_distribution_summary = action_movies.groupby("genre").agg(
    avg_score=("score", "mean"),
    median_score=("score", "median"),
    max_score=("score", "max"),
    min_score=("score", "min"),
    movie_count=("name", "count"),
)

In [ ]:
action_score_distribution_summary

In [ ]:
action_movies

### Question 4: Do higher budeget movies generate more revenue?

In [ ]:
# Average budget, average gross, total gross, and movie count per budget_category (Low, Medium, High, and Very High)
budget_summary = df.groupby("budget_category").agg(
    avg_budget=("budget", "mean"),
    avg_gross=("gross", "mean"),
    total_gross=("gross", "sum"),
    movie_count=("name", "count"),
)

In [ ]:
budget_summary

### Question 5: Which countries produce the most profitable movies?

In [ ]:
# Average return on investment per country
country_roi = df.groupby("country").agg(
    avg_roi=("roi", "mean"),
    movie_count=("name", "count"),
).sort_values("avg_roi", ascending=False)

In [ ]:
country_roi

# Reshaping

In [ ]:
# Create a pivot table showing the mean gross for each (genre, rating) combination
genre_rating_pivot = df.pivot_table(
    values="gross",
    aggfunc="mean",
    index="genre",
    columns="rating",
)

In [ ]:
genre_rating_pivot

# Data Visualization

### Question 1: How had the average rating of R-rated movies changed from 1980-2020?

In [ ]:
import matplotlib.pyplot as plt
import os

fig, ax = plt.subplots()
plt.plot(r_movies_trend.index, r_movies_trend["avg_score"])
plt.title("Average Rating of R-rated Movies Over Time")
plt.xlabel("Year")
plt.ylabel("Average Rating")
plt.grid(True) # Add grid

# Create the 'figures' directory if it doesn't exist
if not os.path.exists('figures'):
    os.makedirs('figures')

plt.savefig('figures/R_rated_movies.png', dpi=300)
plt.show()

### Question 2: Does a certain genre generate high gross revenue?

In [ ]:


#created a violet bar chart and used barh to make it horizontal to fit better
fig, ax = plt.subplots()
plt.barh(genre_gross_summary.index, genre_gross_summary["avg_gross"], color = "violet")
plt.title("Average Gross Revenue by Genre")
plt.xlabel("Genre")
plt.ylabel("Average Gross Revenue")


plt.savefig('figures/grossing_genre.png', dpi=300)
plt.show()

### Question 3: How are IMDb scores distributed across all action movies?

In [ ]:


# created a histogram for distubution with a y grid
fig, ax = plt.subplots()
plt.hist(action_movies["score"], bins=10, edgecolor='black', color='skyblue')
plt.title("Distribution of IMDb Scores for Action Movies")
plt.xlabel("IMDb Score")
plt.ylabel("Number of Movies")
plt.grid(axis='y')
plt.savefig('figures/IMDb_distro.png', dpi=300)
plt.show()

### Question 4: Do higher budget movies generate more revenue?


In [ ]:
import seaborn as sns

#used seaborn with matplot to created a scatterplot with few points anbd made the x and y ticks in a more readable format
fig, ax = plt.subplots()

sns.regplot(x='avg_budget', y='avg_gross', data=budget_summary, color='red')
sns.scatterplot(x='avg_budget', y='avg_gross', data=budget_summary, color = 'black', s = 100 )
plt.title('Average Gross Revenue vs. Average Budget by Category')
plt.xlabel('Average Budget (Per Hundred Million)')
plt.ylabel('Average Gross Revenue (Per Hundred Million)')
plt.grid(True)



plt.savefig('figures/gross_v_rev.png', dpi=300)
plt.show()

# Question 5: Which countries produce the most profitable movies?

In [ ]:
#created another bar chart for Q5
fig, ax = plt.subplots(figsize=(12, 6))

plt.bar(country_roi.index, country_roi["avg_roi"], color="skyblue")
plt.xlabel("Country")
plt.ylabel("Average ROI")
plt.title("Average ROI by Country")
plt.xticks(rotation=90, fontsize=10)
plt.grid(axis='y')
plt.tight_layout()

plt.savefig('figures/country_roi.png', dpi=300)
plt.show()

In [ ]:

# Bonus Question:
# With the data analysis using roi and budget category, it can be interesting to find:
# Which genres are the most consistenlty successful when accounting for both profit and ROI?


In [ ]:
# Bonus feature 1: absolute profit
df["profit"] = df["gross"] - df["budget"]

# Bonus feature 2: success level based on ROI
def success_level(roi):
    if pd.isna(roi):
        return "Unknown"
    elif roi >= 5:
        return "Blockbuster Return"
    elif roi >= 2:
        return "Strong Return"
    elif roi >= 1:
        return "Profitable"
    else:
        return "Loss"

df["success_level"] = df["roi"].apply(success_level)

# Quick check
df[["name", "genre", "budget", "gross", "profit", "roi", "success_level"]].head()

In [ ]:
bonus_genre_summary = df.groupby("genre").agg(
    avg_profit=("profit", "mean"),
    median_profit=("profit", "median"),
    avg_roi=("roi", "mean"),
    movie_count=("name", "count")
).sort_values("avg_profit", ascending=False)

bonus_genre_summary

In [ ]:
success_counts = df.groupby(["genre", "success_level"]).size().unstack(fill_value=0)
success_counts

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

if not os.path.exists("figures"):
    os.makedirs("figures")

# Keeping only the genres with enough movies >=15  so the chart is more balanced and displays a better average
filtered_bonus = bonus_genre_summary[bonus_genre_summary["movie_count"] >= 15].copy()

plt.figure(figsize=(12, 7))
scatter = plt.scatter(
    filtered_bonus["avg_roi"],
    filtered_bonus["avg_profit"],
    s=filtered_bonus["movie_count"] * 8,
    alpha=0.7
)

for genre, row in filtered_bonus.iterrows():
    plt.text(row["avg_roi"], row["avg_profit"], genre, fontsize=9)

plt.title("Genre Success: Average ROI vs Average Profit")
plt.xlabel("Average ROI")
plt.ylabel("Average Profit")
plt.grid(True)
plt.tight_layout()
plt.savefig("figures/bonus_genre_success.png", dpi=300)
plt.show()

In [ ]:
### Bonus Findings

# This bonus analysis goes beyond raw gross revenue by considering both profit and ROI.  
# A genre may earn high total revenue but still be less efficient if its budgets are extremely large.  
# By comparing average profit, average ROI, and movie count together, I can identify which genres are both financially successful and consistently efficient.
# Learned about this graph type using w3schools and AI.